# ಪಾಠ 13 - ಕೋಗಿನ ತಿಳುವಳಿಕೆ ಗ್ರಾಫ್‌ಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ಮೆಮೊರಿ


## ಸೆಟಪ್

ಈ ನೋಟ್ಬುಕ್ [**Cognee**](https://www.cognee.ai/) ಜ್ಞಾನ ಗ್ರಾಫ್‌ಗಳನ್ನು ಮತ್ತು **Microsoft Agent Framework** (MAF) ಬಳಸಿಕೊಂಡು ಸ್ಥಿರ ನೆನಪಿನೊಂದಿಗೆ ಬುದ್ಧಿವಂತ **ಕೋಡಿಂಗ್ ಸಹಾಯಕ** ನಿರ್ಮಿಸುವ ವಿಧಾನವನ್ನು ತೋರಿಸುತ್ತದೆ.

Cognee ರಚನೆಯಲ್ಲದ ಪಠ್ಯವನ್ನು ರಚಿತ, ಪ್ರಶ್ನಿಸಬಹುದಾದ ಜ್ಞಾನ ಗ್ರಾಫ್ ಆಗಿ ಪರಿವರ್ತಿಸುತ್ತದೆ, ಇದು ವೆಕ್ಟರ್ ಎಂಬೆಡ್ಡಿಂಗ್‌ಗಳನ್ನು ಬೆಂಬಲಿಸುತ್ತದೆ — ನಿಮ್ಮ ಏಜಂಟ್‌ಗೆ ಸಮೃದ್ಧ, ಸಂಬಂಧ-ಅಾಹ್ಲಾದಕರ ದೀರ್ಘಕಾಲೀನ ನೆನಪನ್ನು ನೀಡುತ್ತದೆ.

### ನೀವು ಏನು ಕಲಿಯುತ್ತೀರಿ
1. **ಜ್ಞಾನ ಗ್ರಾಫ್‌ಗಳನ್ನು ನಿರ್ಮಿಸಿ**: ಡೆವಲಪರ್ ಪ್ರೊಫೈಲ್ಗಳು ಮತ್ತು ಉತ್ತಮ ಅಭ್ಯಾಸಗಳನ್ನು ರಚಿತ, ಪ್ರಶ್ನಿಸಬಹುದಾದ ಜ್ಞಾನವಾಗಿ ಪರಿವರ್ತಿಸಿ.
2. **Cognee ಅನ್ನು MAF ಜೊತೆಗೆ ಸಂಯೋಜಿಸಿ**: `@tool` ಫಂಕ್ಷನ್‌ಗಳನ್ನು ಬಳಸಿಕೊಂಡು MAF ಏಜಂಟ್ Cognee ಯ ಜ್ಞಾನ ಗ್ರಾಫ್ ಅನ್ನು ಪ್ರಶ್ನಿಸಬಹುದು.
3. **ಸೆಷನ್-ಅವೇರ್ ಸಂವಾದಗಳು**: ಒಟ್ಟاری ಸೆಷನ್‌ನ ಮೇಳದಲ್ಲಿ ಹಲವು ಪ್ರಶ್ನೆಗಳ ನಡುವೆ ಪ್ರಾಸಂಗಿಕತೆ ಕಾಪಾಡಿ.
4. **ದೀರ್ಘಕಾಲೀನ ನೆನಪು**: ಮಹತ್ವದ ಜ್ಞಾಪನೆಯನ್ನು ಸೆಷನ್‌ಗಳ ನಡುವೆ ಉಳಿಸಿ ಮತ್ತು ಹೊಸ ಸಂಭಾಷಣೆಗಳಲ್ಲಿ ಅದನ್ನು ಪಡೆಯಿರಿ.

### ಅವಶ್ಯಕತೆಗಳು
- Python 3.9+
- ಸೆಷನ್ ನಿರ್ವಹಣೆಗೆ ಸ್ಥಳೀಯವಾಗಿ Redis ರನ್ ಮಾಡುತ್ತಿದೆ (`docker run -d -p 6379:6379 redis`)
- LLM API ಕೀ (ಉದಾ. OpenAI) — `.env` ನಲ್ಲಿ `LLM_API_KEY` ಹೊಂದಿಸಿ
- `.env` ನಲ್ಲಿ `CACHING=true` (Cognee ಸೆಷನ್‌ಗಳಿಗೆ ಅಗತ್ಯ)
- ಚಾಟ್ ಮಾದರಿಯನ್ನು ನಿಯೋಜಿಸಿದ Microsoft Foundry ಪ್ರಾಜೆಕ್ಟ್
- `.env` ನಲ್ಲಿ `AZURE_AI_PROJECT_ENDPOINT` ಮತ್ತು `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI ಪ್ರমাণೀಕರಿಸಲಾಗಿದೆ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## ಏಜೆಂಟ್ ಸ್ಮರಣೆ أنواعಗಳು

ಈ ನೋಟ್ಬುಕ್ ಮೆನ್ಸನ್ 13 ನೋಟ್ಬುಕ್‌ನ ಮfactoryನೇ ಮೂರು ಸ್ಮರಣೆ ನstypeಗಳನ್ನು ಪಾರ್ಶ್ವಿಕೆಯಾಗುತ್ತದೆ, ಆದರೆ ದೀರ್ಘಾವಧಿ ಸ್ಮರಣೆ ಬ್ಯಾಕೆಂಡ್ ಆಗಿ Cognee ಅನ್ನು ಬಳಸುತ್ತದೆ:

| ಸ್ಮರಣೆ ಪ್ರಕಾರ | ಯಾಂತ್ರಿಕ ವಿಧಾನ | ಕಾಲಾವಧಿ |
|---|---|---|
| **ಕಾರ್ಯ ನಿರ್ವಹಣೆ** | `agent.create_session()` (MAF) | ಒಂದು ಸಂಭಾಷಣೆ |
| **ಸಣ್ಣಕಾಲೀನ** | Cognee ಸೆಷನ್ ಕೇಶ್ (Redis) | ಒಂದು ಸೆಷನ್ |
| **ದೀರ್ಘಕಾಲೀನ** | Cognee ಜ್ಞಾನ ಗ್ರಾಫ್ + ವೆಕ್ಟರ್‌ಗಳು | ಶಾಶ್ವತ |

### Cognee ನ ಸ್ಮರಣೆ ವಾಸ್ತುಶಿಲ್ಪ
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Cognee ಸಂಗ್ರಹಣೆಯನ್ನು ಸಿದ್ಧಪಡಿಸಿ


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## ಭಾಗ 1 — ಜ್ಞಾನ ಆಧಾರದ ನಿರ್ಮಾಣ

ನಮ್ಮ ಕೋಡಿಂಗ್ ಸಹಾಯಕರಿಗಾಗಿ ಸಮಗ್ರ ಜ್ಞಾನ ಆಧಾರವನ್ನು ರಚಿಸಲು ನಾವು ಮೂರು ವಿಧದ ಡೇಟಾವನ್ನು ಸೇರಿಸಿಕೊಂಡಿದ್ದೇವೆ:

1. **ಡೆವಲಪರ್ ಪ್ರೊಫೈಲ್** — ವೈಯಕ್ತಿಕ ಪರಿಣತಿ ಮತ್ತು ತಾಂತ್ರಿಕ ಹಿನ್ನೆಲೆ
2. **ಪೈಥನ್ ಮಹತ್ವಪೂರ್ಣ ಅಭ್ಯಾಸಗಳು** — ಪೈಥನ್ ನ ತತ್ವಜ್ಞಾನ ಹಾಗೂ ನೈಜ ಮಾರ್ಗದರ್ಶಕಗಳು
3. **ಇತಿಹಾಸದ ಸಂಭಾಷಣೆಗಳು** — ಅಭಿವೃದ್ಧಿಪಡಿಸುವವರ ಮತ್ತು AI ಸಹಾಯಕರ ನಡುವಿನ ಹಳೆಯ ಪ್ರಶ್ನೋತ್ತರ ಅಧಿವೇಶನಗಳು


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ಜ್ಞಾನ ಗ್ರಾಫ್ ಅನ್ನು ದೃಶ್ಯೀಕರಿಸಿ

Cognee ಇದು ಹೊರತೆಗೆಯಲಾದ ಘಟಕಗಳು ಮತ್ತು ಸಂಪರ್ಕಗಳ ಸಂವಹನಾತ್ಮಕ HTML ದೃಶ್ಯೀಕರಣವನ್ನು ರೆಂಡರ್ ಮಾಡಬಹುದು.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## ಮೆಮೊರಿ ಮಿತಿಫೈ ಮೂಲಕ ಶ್ರೀಮಂತಗೊಳಿಸಿ

`memify()` ಜ್ಞಾನ ಗ್ರಾಫ್ ಅನ್ನು ವಿಶ್ಲೇಷಿಸಿ ಬುದ್ಧಿವಂತ ನಿಯಮಗಳನ್ನು ರಚಿಸುತ್ತದೆ — ಮಾದರಿಗಳು, ಅತ್ಯುತ್ತಮ ಅಭ್ಯಾಸಗಳು ಮತ್ತು ಕನ್ಸೆಪ್ಟ್‌ಗಳ ನಡುವಿನ ಸಂಬಂಧಗಳನ್ನು ಗುರುತಿಸುತ್ತದೆ.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## ಭಾಗ 2 — ಕೊಗ್ನೀ ಸಾಧನಗಳೊಂದಿಗೆ MAF ಏಜೆಂಟ್  

ಈಗ ನಾವು `@tool` ಕಾರ್ಯಗಳ ಮೂಲಕ ಕೊಗ್ನೀ ಜ್ಞಾನ ಗ್ರಾಫ್ ಅನ್ನು ಪ್ರಶ್ನಿಸಲು ಸಾಧ್ಯವಾಗುವ MAF ಏಜೆಂಟ್ ಅನ್ನು ನಿರ್ಮಿಸುತ್ತೇವೆ. ಇದು ಏಜೆಂಟ್‌ಗೆ ಸಂವಾದ ಸತ್ರಗಳ ಮೂಲಕ ಸಂವಾದಾತ್ಮಕ ಸੰਦਰಭವನ್ನು ಕಾಯ್ದು ಕೊಳ್ಳುವ ಜೊತೆಗೆ ಗ್ರಾಫ್-ಜ್ಞಾನಾರ್ಹ ಅರ್ಥ Searches ಸಂಪೂರ್ಣ ಶಕ್ತಿಯನ್ನು ಬಳಸಿಕೊಳ್ಳಲು ಅನುವು ಮಾಡಿಕೊಡುತ್ತದೆ.  


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## ಸೆಷನ್‌ಗಳೊಂದಿಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುವ ಮೆಮೊರಿ

`agent.create_session()` ಮೂಲಕ ರಚಿಸಲಾದ `AgentSession` ಸೆಷನ್ ಒಳಗಿನ ಕಾರ್ಯನಿರ್ವಹಿಸುವ ಮೆಮೊರಿಯನ್ನು ಒದಗಿಸುತ್ತದೆ. ಏಜೆಂಟ್ ಹಿಂದಿನ ಸಂದೇಶಗಳನ್ನು ಹಿಂತಿರುಗಿ ನೋಡಬಹುದು ಮತ್ತು Cognee ಯ ದೀರ್ಘಕಾಲೀನ ಜ್ಞಾನ ಗ್ರಾಫ್ ಅನ್ನು ಕೂಡ ಪ್ರಶ್ನಿಸಬಹುದು.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## ಹೊಸ ಸೆಷನ್ — ದೀರ್ಘಕಾಲಿಕ ಸ್ಮರಣೆ ಉಳಿದುಕೊಳ್ಳುತ್ತದೆ

ಹೊಸ ಸೆಷನ್ ಪ್ರಾರಂಭಿಸುವುದು ಕಾರ್ಯನಿರ್ವಹಣಾ ಸ್ಮರಣೆಯನ್ನು ಶುಚಿತ್ವಗೊಳಿಸುತ್ತದೆ, ಆದರೆ Cognee ಜ್ಞಾನ ಗ್ರಾಫ್ ಇನ್ನೂ ಲಭ್ಯವಿದೆ. ಏಜೆಂಟ್ ಸಂಪೂರ್ಣ ಹೊಸ ಸಂಭಾಷಣದಲ್ಲಿ ಅದೇ ದೀರ್ಘಕಾಲಿಕ ಜ್ಞಾನದ retrieval ಸಾಧ್ಯವಾಗುತ್ತದೆ.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

In this notebook you built a coding assistant that combines **MAF's working memory** (`agent.create_session()`) with **Cognee's long-term knowledge graph**.

### What You've Learned
1. **Knowledge graph construction**: Cognee ingests unstructured text and builds a graph + vector memory.
2. **Graph enrichment with memify**: Derived facts and richer relationships on top of your existing graph.
3. **MAF + Cognee integration**: `@tool` functions let an MAF agent query Cognee's graph naturally.
4. **Working memory + long-term memory**: `AgentSession` (via `agent.create_session()`) provides session context while Cognee provides persistent knowledge.
5. **Filtered search with NodeSets**: Target specific subsets of the knowledge graph (e.g. only principles).

### Key Takeaways
- **Cognee** turns raw text into structured, relationship-aware memory — more powerful than a flat vector store.
- **`@tool` functions** bridge MAF agents and external knowledge systems cleanly.
- **`AgentSession`** (via `agent.create_session()`) keeps per-conversation context separate from long-lived knowledge.
- The same knowledge graph serves multiple sessions and agents.

### Real-World Applications
- **Developer copilots**: Code review, incident analysis, architecture assistants
- **Customer-facing copilots**: Support agents over product docs, FAQs, and CRM notes
- **Internal expert copilots**: Policy, legal, or security assistants reasoning over guidelines
- **Unified data layers**: Combine structured and unstructured data into one queryable graph

### Next Steps
- Experiment with temporal awareness in Cognee
- Define an OWL ontology for domain-specific graph quality
- Add user feedback loops to improve retrieval over time
- Scale to multi-agent systems sharing the same Cognee memory layer


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
